Imported process to re-run NLP in databricks

In [0]:
# %restart_python
%pip install "pandas==1.5.3" "pyarrow==15.0.2" "openpyxl==3.1.5" "loguru==0.7.3"
# %pip show pandas pyarrow openpyxl loguru

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from loguru import logger
# Added logger for the notebook
logger.remove()
logger.add(lambda msg: print(msg, end=""))
logger.add("/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2/logs/error.logs", level="ERROR")
logger.add("/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2/logs/processing.log", level="DEBUG")



def get_patient_mrn_paths(input_file: str) -> Tuple[Path, Path, Path]:
    current_effort_root = "/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/update_with_new_mrns"
    data_root = Path(current_effort_root) / "data"
    patient_mrn_data_path = Path(data_root) / "patient_mrns"
    input_path = patient_mrn_data_path / input_file
    output_path = patient_mrn_data_path / "mrns.parquet"
    dropped_patients__duplicate_dedup_ids_path = patient_mrn_data_path / f"dropped_patients__duplicate_dedup_ids__{input_file}"

    return input_path, output_path, dropped_patients__duplicate_dedup_ids_path

1

In [0]:
import pandas as pd
import os
from pathlib import Path
from typing import Tuple
from loguru import logger

input_path, output_path, dropped_patients__duplicate_dedup_ids_path = \
    get_patient_mrn_paths("0.0.2_p6290_xwalk_mrn_deid.csv")

def build_mrn_input(_output_path: Path):
    if os.path.exists(_output_path):
        logger.info("{} path already exists; to recreate, delete existing data set.")
    else:
        logger.info("Building MRN input data set...")
        # load dedup ids -> MRNs
        df = pd.read_csv(input_path, dtype={'deid_pat_id': 'int32', 'mrn': 'int64'})
        logger.info("{:,} entries loaded", len(df))
        df = df.reset_index(names=['index'])
        
        # Identify all duplicates
        duplicate_indices = df.index[df['deid_pat_id'].duplicated(keep=False)]
        duplicates_df = df[df['deid_pat_id'].isin(df['deid_pat_id'][duplicate_indices])]
        
        # Get a test case
        test_deid_pat_id = int(duplicates_df.iloc[0][['deid_pat_id']])
        test_mask_duplicates = duplicates_df['deid_pat_id'] == test_deid_pat_id
        test_mask_df = df['deid_pat_id'] == test_deid_pat_id
        assert len(df[test_mask_df]) > 1

        # Keep first entry for each duplicate
        deduped_df = df.drop_duplicates(subset=['deid_pat_id'], keep='first')
        logger.info('{:,} unique deid_pat_id entries kept', len(deduped_df))

        assert len(deduped_df[deduped_df['deid_pat_id'] == test_deid_pat_id]) == 1
        
        rows_to_drop_df = df[~df['index'].isin(deduped_df['index'])]
        logger.info('{:,} duplicates to drop', len(rows_to_drop_df))
        
        # Persist all the duplicates not processed
        rows_to_drop_df.to_csv(dropped_patients__duplicate_dedup_ids_path, index=False)
        del rows_to_drop_df
        del df
        
        # Persist the deduped values to process
        deduped_df.to_parquet(output_path, engine="pyarrow")
        del deduped_df

        # verify load
        df_pd = pd.read_parquet(output_path)
        assert not df_pd['deid_pat_id'].duplicated().any(), "Duplicates found in deid_pat_id"
        assert not df_pd['mrn'].duplicated().any(), "Duplicates found in mrn"
        logger.info("Verified no duplicates present.")
        del df_pd
        logger.success("MRN input data set built.")

build_mrn_input(output_path)
        

2025-11-11 16:22:32.070 | INFO     | __main__:build_mrn_input:14 - Building MRN input data set...
2025-11-11 16:22:33.066 | INFO     | __main__:build_mrn_input:17 - 1,083,920 entries loaded
2025-11-11 16:22:33.119 | INFO     | __main__:build_mrn_input:32 - 1,083,853 unique deid_pat_id entries kept
2025-11-11 16:22:33.146 | INFO     | __main__:build_mrn_input:37 - 67 duplicates to drop
2025-11-11 16:22:35.365 | INFO     | __main__:build_mrn_input:52 - Verified no duplicates present.
2025-11-11 16:22:35.366 | SUCCESS  | __main__:build_mrn_input:54 - MRN input data set built.
